In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv")

# ============================================================
# 1. STATE → TÜRKİYE İL MAPPING
# ============================================================
state_to_turkey = {
    'California': 'Istanbul',
    'Washington': 'Ankara',
    'Arizona':    'Izmir',
    'Nevada':     'Bursa',
    'Oregon':     'Antalya'
}
df['City'] = df['State'].map(state_to_turkey)

# Bölgesel Trafik Risk Skoru (1-5 arası)
city_risk_score = {
    'Istanbul': 5,
    'Ankara':   4,
    'Izmir':    4,
    'Bursa':    3,
    'Antalya':  3
}
df['city_risk_score'] = df['City'].map(city_risk_score)

# ============================================================
# 2. YEDEK PARÇA / KUR ÇARPANI (parts_inflation_index)
# ============================================================
luxury_vehicles = ['Luxury SUV', 'Luxury Car', 'Sports Car']

df['parts_inflation_index'] = np.where(
    df['Vehicle Class'].isin(luxury_vehicles), 1.5, 1.0
)

# ============================================================
# 3. HASARSIZLIK SKORU (no_claim_score)
# Months Since Last Claim ne kadar büyükse, o kadar "iyi" sürücü
# ============================================================
df['no_claim_score'] = pd.cut(
    df['Months Since Last Claim'],
    bins=[-1, 6, 12, 24, 99],
    labels=[1, 2, 3, 4]   # 1=kötü, 4=çok iyi
).astype(int)

# ============================================================
# 4. GELİR RİSK ORANI (income_risk_ratio)
# Düşük gelir + yüksek prim = ödeme riski
# ============================================================
df['income_risk_ratio'] = np.where(
    df['Income'] > 0,
    df['Monthly Premium Auto'] / (df['Income'] / 12),
    df['Monthly Premium Auto']  # geliri 0 olanlar için direkt prim
)

# ============================================================
# 5. LOCATİON CODE → SAYISAL
# ============================================================
location_map = {'Urban': 3, 'Suburban': 2, 'Rural': 1}
df['location_score'] = df['Location Code'].map(location_map)

# ============================================================
# KONTROL
# ============================================================
print("Yeni kolonlar:")
print(df[['City', 'city_risk_score', 'parts_inflation_index', 
          'no_claim_score', 'income_risk_ratio', 'location_score']].head(10))

print("\nYeni kolon istatistikleri:")
print(df[['city_risk_score', 'parts_inflation_index', 
          'no_claim_score', 'income_risk_ratio']].describe())
# ============================================================
# income_risk_ratio OUTLIER DÜZELTMESİ
# Log dönüşümü + cap uygula
# ============================================================

# Önce 99. percentile değerini bul
cap_value = df['income_risk_ratio'].quantile(0.99)
print(f"Cap değeri (99. percentile): {cap_value:.2f}")

# 99. percentile üzerindeki değerleri kırp
df['income_risk_ratio'] = df['income_risk_ratio'].clip(upper=cap_value)

# Log dönüşümü uygula (dağılımı normalize et)
df['income_risk_ratio_log'] = np.log1p(df['income_risk_ratio'])

# Kontrol
print("\nDüzeltme sonrası income_risk_ratio:")
print(df['income_risk_ratio'].describe())
print("\nLog dönüşümü sonrası:")
print(df['income_risk_ratio_log'].describe())


# ============================================================
# KAYDET
# ============================================================
df.to_csv("../data/processed/intelliprice_featured.csv", index=False)
print("\n✅ Veri 'processed' klasörüne kaydedildi!")

Yeni kolonlar:
       City  city_risk_score  parts_inflation_index  no_claim_score  \
0    Ankara                4                    1.0               4   
1     Izmir                4                    1.0               3   
2     Bursa                3                    1.0               3   
3  Istanbul                5                    1.0               3   
4    Ankara                4                    1.0               2   
5   Antalya                3                    1.0               3   
6   Antalya                3                    1.0               1   
7     Izmir                4                    1.0               1   
8   Antalya                3                    1.0               3   
9   Antalya                3                    1.0               3   

   income_risk_ratio  location_score  
0           0.014714               2  
1          94.000000               2  
2           0.026575               2  
3         106.000000               2  
4       